# Aprendizado de Máquina — Lista prática 10

## KNN e Árvores de Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Helena é estatística no laboratório de patologia de um hospital universitário. O laboratório
quer um classificador que ajude a separar tumores de mama malignos de benignos a partir de
medidas feitas em imagens de células, e Helena vai experimentar os dois métodos da Aula 10.
Antes de olhar dados reais, ela estuda o KNN em dados simulados, em que sabe como as classes
foram geradas. Depois, ajusta árvores de classificação ao `breast_cancer` do `scikit-learn`, um
banco público de 569 tumores, cada um descrito por 30 medidas de imagem e classificado como
benigno ou maligno. Os exercícios seguem a Aula prática 10: o primeiro corresponde à Seção 2, e
os outros dois, à Seção 4.

Cada lacuna está marcada com `...`. Substitua cada uma pela sua resposta e rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — o $k$ do KNN

Helena começa por uma população de duas meias-luas entrelaçadas, geradas pelo `make_moons`, uma
por classe, com ruído normal de desvio-padrão 0,3 em cada coordenada. A amostra de treino repete
as contagens do Exercício 2 da Lista teórica 10: $n = 301$ observações, 181 da classe 1 e 120 da
classe 0 (o argumento `n_samples=(120, 181)` dá as quantidades da classe 0 e da classe 1, nessa
ordem). Para medir o desempenho em observações novas, ela sorteia outras 10 000 da mesma
população, 60% da classe 1, como no treino.

Ela ajusta o KNN com seis valores de $k$, dos extremos $k = 1$ e $k = n$ a valores
intermediários, e compara a acurácia de treino com a acurácia nas observações novas. (a) Passe a
cada ajuste o número de vizinhos do laço. (b) Calcule a acurácia nas observações novas. A
célula imprime as duas acurácias e as desenha contra $k$, em escala logarítmica.

In [ ]:
X_tr, y_tr = make_moons(n_samples=(120, 181), noise=0.3, random_state=2026)
X_te, y_te = make_moons(n_samples=(4000, 6000), noise=0.3, random_state=2027)
n = len(y_tr)
print(f"treino: {n} observacoes, {y_tr.sum()} da classe 1; observacoes novas: {len(y_te)}")
print(f"as {n} observacoes de treino sao distintas? {len(np.unique(X_tr, axis=0)) == n}\n")

valores_k = (1, 5, 15, 51, 151, n)
acc_treino, acc_novas = [], []
print("     k    treino    teste")
for k in valores_k:
    modelo = KNeighborsClassifier(n_neighbors=...).fit(X_tr, y_tr)          # (a)
    acc_treino.append(modelo.score(X_tr, y_tr))
    acc_novas.append(...)                                                    # (b)
    print(f"  {k:4d}   {acc_treino[-1]:.4f}   {acc_novas[-1]:.4f}")

fig, ax = subplots(figsize=(5.6, 3.4))
ax.plot(valores_k, acc_treino, "o-", label="treino")
ax.plot(valores_k, acc_novas, "o-", label="observacoes novas")
ax.set_xscale("log")
ax.set_xticks(valores_k, [str(k) for k in valores_k])
ax.set_xlabel("k (escala logaritmica)"); ax.set_ylabel("acuracia")
ax.legend(fontsize=8);

Na prática, Helena não teria as 10 000 observações novas para escolher $k$. Ela usa, então,
validação cruzada com 10 dobras estratificadas, nos valores ímpares de $k$ de 1 a 151, e só
depois confere o $k$ escolhido nas observações novas. (c) Passe ao `GridSearchCV` o dicionário
com a grade de valores de `n_neighbors`. (d) Calcule a acurácia do $k$ escolhido nas observações
novas. A última parte da célula imprime, para quatro valores de $k$, a acurácia de validação
cruzada e a de teste, e desenha as duas para todos os valores de $k$ da grade.

In [ ]:
cv = skm.StratifiedKFold(10, shuffle=True, random_state=2026)
grade = np.arange(1, 152, 2)
busca = skm.GridSearchCV(KNeighborsClassifier(), {"n_neighbors": ...}, cv=cv).fit(X_tr, y_tr)   # (c)
print(f"k escolhido: {busca.best_params_['n_neighbors']}, acuracia de validacao cruzada "
      f"{busca.best_score_:.4f}, nas observacoes novas {...:.4f}")                          # (d)

media = busca.cv_results_["mean_test_score"]
perto = grade[media >= media.max() - 0.01]
print(f"valores de k a menos de 1 ponto do maximo da validacao cruzada: {len(perto)} de {len(grade)}\n")
print("     k    validacao cruzada    teste")
for k in (1, 5, 15, busca.best_params_["n_neighbors"]):
    teste = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr).score(X_te, y_te)
    print(f"  {k:4d}        {media[grade == k][0]:.4f}          {teste:.4f}")

acc_grade = [KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr).score(X_te, y_te) for k in grade]
fig, ax = subplots(figsize=(6.2, 3.4))
ax.plot(grade, media, label="validacao cruzada (10 dobras)")
ax.plot(grade, acc_grade, label="observacoes novas")
ax.axvline(busca.best_params_["n_neighbors"], ls=":", color="gray", label="k escolhido")
ax.set_xlabel("k"); ax.set_ylabel("acuracia"); ax.legend(fontsize=8);

> **Sua vez.** Desenhe as fronteiras de decisão do KNN com $k = 1$, $k = 5$ e $k = n$, como faz
> a função `desenhar` da Aula prática 10: use `np.meshgrid` para gerar uma grade fina do plano,
> `modelo.predict` sobre ela, `ax.contourf(...)` para pintar o fundo com a classe prevista e
> `ax.contour(..., levels=[0.5], colors="k")` para traçar a fronteira; por cima, desenhe as
> observações de treino. Onde fica a fronteira com $k = n$?

---
## Exercício 2 — Gini ou entropia?

Helena passa aos tumores. No `breast_cancer`, a classe 1 é a dos tumores benignos, 357 dos 569,
e a classe 0, a dos malignos. Ela vai crescer árvores de classificação e precisa escolher a
medida de impureza. O Exercício 1 da Lista teórica 10 mostrou que o erro de classificação pode
registrar redução zero num corte que separa uma região pura, e o `DecisionTreeClassifier` só
oferece, para crescer, o Gini e a entropia. Helena quer saber se a escolha entre os dois
importa. Ela estima, por validação cruzada com 5 dobras estratificadas, a acurácia das árvores
com cada medida e com profundidade máxima 3, 5 e sem limite. (a) e (b) Escreva os valores de
`criterion` que o `DecisionTreeClassifier` usa para o Gini e para a entropia. A célula imprime
as seis acurácias e as desenha contra a profundidade.

In [ ]:
dados = load_breast_cancer()
X_bc, y_bc = dados.data, dados.target
cv = skm.StratifiedKFold(5, shuffle=True, random_state=2026)

criterios = (..., ...)                                                    # (a) e (b)
acc = {}
for criterio in criterios:
    for prof in (3, 5, None):
        arvore = DecisionTreeClassifier(criterion=criterio, max_depth=prof, random_state=0)
        acc[criterio, prof] = skm.cross_val_score(arvore, X_bc, y_bc, cv=cv).mean()
        print(f"  {criterio:8s} max_depth={str(prof):4s}: acuracia (CV) {acc[criterio, prof]:.4f}")

fig, ax = subplots(figsize=(5, 3.4))
for criterio in criterios:
    ax.plot(["3", "5", "sem limite"], [acc[criterio, p] for p in (3, 5, None)], "o-", label=criterio)
ax.set_xlabel("max_depth"); ax.set_ylabel("acuracia (validacao cruzada)")
ax.legend(fontsize=8);

A partição dos 569 tumores em 5 dobras depende da semente do `StratifiedKFold`. (c) Repita a
comparação com 20 partições, das sementes 0 a 19, e conte, em cada profundidade, em quantas o
Gini fica à frente. A célula imprime também em quantas partições a árvore de profundidade 5 fica
à frente da de profundidade 3, com o Gini, e a faixa de acurácias da árvore de profundidade 5
entre as partições. Leva alguns segundos. A figura põe cada partição num ponto: em azul, a
diferença de acurácia entre o Gini e a entropia em cada profundidade; em laranja, a diferença
entre as profundidades 5 e 3, com o Gini.

In [ ]:
dif_criterio = {3: [], 5: [], None: []}    # Gini menos entropia, em cada particao
dif_prof, acc5 = [], []                    # profundidade 5 menos 3, com o Gini
for semente in range(20):
    cv_s = skm.StratifiedKFold(5, shuffle=True, random_state=...)                         # (c)
    acc_s = {(c, p): skm.cross_val_score(DecisionTreeClassifier(criterion=c, max_depth=p, random_state=0),
                                         X_bc, y_bc, cv=cv_s).mean()
             for c in ("gini", "entropy") for p in (3, 5, None)}
    for p in (3, 5, None):
        dif_criterio[p].append(acc_s[("gini", p)] - acc_s[("entropy", p)])
    dif_prof.append(acc_s[("gini", 5)] - acc_s[("gini", 3)])
    acc5.append(acc_s[("gini", 5)])

for p in (3, 5, None):
    print(f"max_depth={str(p):4s}: o Gini fica a frente em {np.sum(np.array(dif_criterio[p]) > 0):2d} das 20 particoes")
print(f"\ncom o Gini, a profundidade 5 fica a frente da 3 em {np.sum(np.array(dif_prof) > 0)} das 20 particoes")
print(f"acuracia da arvore de profundidade 5 com o Gini: de {min(acc5):.4f} a {max(acc5):.4f}")

fig, ax = subplots(figsize=(7, 3.6))
grupos = [dif_criterio[3], dif_criterio[5], dif_criterio[None], dif_prof]
rotulos = ["Gini - entropia\nmax_depth=3", "Gini - entropia\nmax_depth=5",
           "Gini - entropia\nsem limite", "profundidade 5 - 3\n(Gini)"]
espalhar = np.linspace(-0.15, 0.15, 20)    # afasta os 20 pontos de cada grupo, para nao se sobreporem
for i, grupo in enumerate(grupos):
    ax.scatter(i + espalhar, grupo, s=14, color="steelblue" if i < 3 else "darkorange")
ax.axhline(0, ls="--", color="gray", lw=1)
ax.set_xticks(range(4), rotulos, fontsize=7.5)
ax.set_ylabel("diferenca de acuracia (CV)");

Por fim, Helena confere que o erro de classificação não está entre os critérios aceitos. (d)
Passe ao `criterion` um nome para o erro de classificação.

In [ ]:
try:
    DecisionTreeClassifier(criterion=...).fit(X_bc, y_bc)                  # (d)
except Exception as erro:
    print(type(erro).__name__)
    print(str(erro)[:160])

---
## Exercício 3 — a poda por `ccp_alpha`

Helena separa 30% dos tumores para teste, preservando a proporção das classes, e cresce uma
árvore com o Gini, sem limite de profundidade, nos outros 70%. Para podá-la, usa o caminho de
custo–complexidade do `scikit-learn`, como na Seção 4 da Aula prática 10: o
`cost_complexity_pruning_path` devolve os valores de $\alpha$ em que a subárvore ótima muda
(`ccp_alphas`) e, para cada um, a soma das impurezas das folhas dessa subárvore, ponderadas por
$n_m/n$ (`impurities`). O último $\alpha$ da lista poda a árvore até a raiz e fica fora da
busca. (a) Calcule o número de folhas da árvore cheia. (b) Escolha $\alpha$ por validação
cruzada com 10 dobras estratificadas, comparando as subárvores pela acurácia. A célula desenha
a subárvore escolhida com o `plot_tree`, que mostra, em cada nó, o corte, o Gini (`gini`), o
número de tumores (`samples`) e as contagens das duas classes (`value`).

In [ ]:
X_tr, X_te, y_tr, y_te = skm.train_test_split(X_bc, y_bc, test_size=0.3,
                                              random_state=2026, stratify=y_bc)
cheia = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
print(f"arvore cheia: {...} folhas, acuracia de treino {cheia.score(X_tr, y_tr):.4f}, "      # (a)
      f"de teste {cheia.score(X_te, y_te):.4f}")

caminho = cheia.cost_complexity_pruning_path(X_tr, y_tr)
alphas = caminho.ccp_alphas[:-1]
cv10 = skm.StratifiedKFold(10, shuffle=True, random_state=2026)
busca = skm.GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": alphas},
                         cv=cv10, scoring=...).fit(X_tr, y_tr)                           # (b)
podada = busca.best_estimator_
print(f"arvore podada: alpha = {busca.best_params_['ccp_alpha']:.5f}, {podada.get_n_leaves()} folhas, "
      f"acuracia de validacao cruzada {busca.best_score_:.4f}, de teste {podada.score(X_te, y_te):.4f}")

fig, ax = subplots(figsize=(13, 6))
plot_tree(podada, feature_names=list(dados.feature_names), class_names=["maligno", "benigno"],
          ax=ax, fontsize=7)
ax.set_title(f"a subarvore podada: {podada.get_n_leaves()} folhas", fontsize=10);

O `scikit-learn` poda pela impureza das folhas, não pelo erro de classificação. (c) Confira isso
na subárvore escolhida: calcule a soma ponderada dos Gini das folhas, $\sum_m (n_m/n)\,G(R_m)$,
com os atributos `n_node_samples` e `impurity` do `tree_`, e compare-a com o `impurities` do
caminho e com a taxa de erro de treino da subárvore. Os números de cada folha também estão no
desenho do item (b).

In [ ]:
arv = podada.tree_
folha = arv.children_left == -1                    # as folhas nao tem filhos
n_m, gini_m = arv.n_node_samples[folha], arv.impurity[folha]
soma = ...                                                                     # (c)
i = np.flatnonzero(alphas == busca.best_params_["ccp_alpha"])[0]
print(f"soma ponderada dos Gini das folhas: {soma:.4f}")
print(f"impurities do caminho              : {caminho.impurities[i]:.4f}")
print(f"taxa de erro de treino             : {1 - podada.score(X_tr, y_tr):.4f}")

Helena desconfia da escolha: a validação cruzada comparou dez subárvores com 398 tumores. A
célula a seguir imprime, para cada subárvore da sequência, o número de folhas, a acurácia de
validação cruzada com as 10 dobras do item (b), a acurácia de validação cruzada com 5 dobras e a
acurácia de teste, e desenha as três contra o número de folhas. (d) Ajuste, para cada
$\alpha$, a subárvore correspondente.

In [ ]:
cv5 = skm.StratifiedKFold(5, shuffle=True, random_state=2026)
acc10 = busca.cv_results_["mean_test_score"]
folhas, acc5_sub, acc_teste = [], [], []
print(" folhas   CV (10 dobras)   CV (5 dobras)   teste")
for a, m10 in zip(alphas, acc10):
    subarvore = DecisionTreeClassifier(random_state=0, ccp_alpha=...).fit(X_tr, y_tr)       # (d)
    m5 = skm.cross_val_score(DecisionTreeClassifier(random_state=0, ccp_alpha=a), X_tr, y_tr, cv=cv5).mean()
    folhas.append(subarvore.get_n_leaves()); acc5_sub.append(m5); acc_teste.append(subarvore.score(X_te, y_te))
    print(f"  {folhas[-1]:3d}        {m10:.4f}          {m5:.4f}        {acc_teste[-1]:.4f}")

fig, ax = subplots(figsize=(6.4, 3.6))
ax.plot(folhas, acc10, "o-", ms=4, label="validacao cruzada, 10 dobras")
ax.plot(folhas, acc5_sub, "s-", ms=4, label="validacao cruzada, 5 dobras")
ax.plot(folhas, acc_teste, "^-", ms=4, label="teste (171 tumores)")
ax.set_xlabel("numero de folhas"); ax.set_ylabel("acuracia"); ax.legend(fontsize=8);

> **Sua vez.** Repita o item (b) com as sementes de 0 a 9 no `StratifiedKFold` de 10 dobras e
> anote, para cada uma, o número de folhas da subárvore escolhida e a acurácia dela nos tumores
> de teste. Quantas subárvores diferentes aparecem? O que isso diz sobre a precisão da validação
> cruzada com 398 tumores?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | $k = 1$ acerta todo o treino; $k = n = 301$ responde sempre a classe 1 e acerta $181/301$ do treino, como no Exercício 2(a) da Lista teórica 10 |
| 1 | a validação cruzada escolhe $k = 63$, que acerta 0,8959 das observações novas, contra 0,9135 de $k = 5$: com 301 observações, ela não separa os dois |
| 2 | numa partição, o Gini fica à frente nas três profundidades; em 20 partições, em 4, 9 e 4 delas, enquanto a profundidade 5 vence a 3 em 16 |
| 2 | o erro de classificação não é um `criterion` aceito, e `'entropy'` e `'log_loss'` são a mesma medida |
| 3 | o `ccp_alpha` poda pela soma ponderada dos Gini das folhas; o erro de classificação entra na escolha de $\alpha$ pela validação cruzada |
| 3 | a subárvore de 9 folhas acerta 161 dos 171 tumores de teste, contra 155 da árvore cheia; com 5 dobras, a escolha seria a de 3 folhas, que acerta 153 |

**A seguir.** A Aula 10 encerra a parte de classificação do curso. As aulas extras tratam de
outros problemas: a E1 e a E2, de dados sem resposta, com o agrupamento e a redução de
dimensionalidade, e a E3 aplica classificadores a textos.